# TOD export em background + widget

Este notebook demonstra o fluxo não-bloqueante de simulação TOD (tickets TOD-003 → TOD-007):

- `run_simulation(..., background=True)` retorna um `JobHandle` imediatamente;
- `handle.wait()` acompanha até o fim (SUCCESS/FAILED/CANCELLED);
- `progress_widget(handle)` renderiza barra de progresso, etapa, ETA e botão de cancelar.

> Requer `ipywidgets` instalado e um `settings.toml` com perfil de telescópio (ex.: `uirapuru`).

In [ ]:
from radioskysims.interfaces.notebook_api import (
    JobHandle,
    progress_widget,
    run_simulation,
)

## 1. Iniciar job em background

O retorno é imediato (o worker roda em subprocesso).

In [ ]:
handle = run_simulation(
    "settings.toml",
    telescope="uirapuru",
    constellations=["GPS"],
    use_processor=True,
    background=True,
)
print(handle)
assert isinstance(handle, JobHandle)

## 2. Acompanhar o progresso

`handle.refresh()` lê o banco de jobs + arquivo JSON de progresso. O widget abaixo atualiza sozinho e tem botão **Cancelar**.

In [ ]:
progress_widget(handle, interval=2.0)

## 3. Aguardar o fim (bloqueante)

`wait()` retorna um dicionário com o snapshot final.

In [ ]:
snap = handle.wait(timeout=600, interval=2.0)
print("status:", snap["status"])
print("percent:", snap["percent"])
print("step:", snap["step"])
print("output:", snap["output_path"])
print("erro:", snap["error"] or "-")

## 4. Cancelar (opcional)

Se o job ainda estiver `RUNNING`, `cancel()` envia SIGTERM ao worker e grava `CANCELLED` no banco.

In [ ]:
# handle.cancel()  # descomente para cancelar um job em execução